# Pakistan House Price Predictor: Data Cleaning and EDA

**Project 1** | Goal: predict house sale prices in Pakistani cities.

**Notebook 01:** load the data, clean it, and save a cleaned file for modelling.

## 1. Dataset Information

- **Source:** [Pakistan House Price Dataset (Kaggle)](https://www.kaggle.com/datasets/jillanisofttech/pakistan-house-price-dataset), based on Zameen.com listings
- **Size:** 168,446 rows and 20 columns
- **Data types:**
  - int64 (5): property_id, location_id, price, baths, bedrooms
  - float64 (3): latitude, longitude, Area Size
  - object (12): page_url, property_type, location, city, province_name, area, purpose, date_added, agency, agent, Area Type, Area Category
- **Missing values:** `agency` has 44,071 (about 26%) and `agent` has 44,072 (about 26%). All other columns have no NaN values.

### Column descriptions

| Column | Meaning | Type |
|---|---|---|
| property_id | Unique ID of the listing | int |
| location_id | ID of the locality | int |
| page_url | Link to the Zameen listing | object |
| property_type | House, Flat, Upper Portion, Lower Portion, Room, Farm House, Penthouse | object |
| price | Listed price in PKR | int |
| location | Neighbourhood or society | object |
| city | City | object |
| province_name | Province | object |
| latitude, longitude | Map coordinates | float |
| baths | Number of bathrooms | int |
| area | Area as text | object |
| purpose | For Sale or For Rent | object |
| bedrooms | Number of bedrooms | int |
| date_added | Date the listing was posted (month-day-year) | object |
| agency, agent | Seller's agency and agent | object |
| Area Type | Unit of the area: Marla or Kanal | object |
| Area Size | Area value in the unit given by Area Type | float |
| Area Category | Size bucket such as "0-5 Marla" | object |

## 2. Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/zameen-updated.csv")
rows_before = len(df)          # starting size, used in the final report
summary = {}                   # rows removed at each cleaning step

print("Data shape:", df.shape)
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

## 3. Initial checks

Before changing anything, I check three things that can silently ruin a price model: mixed purposes (sale vs rent), mixed area units, and duplicate listings.

In [ ]:
df["purpose"].value_counts()

In [ ]:
df["Area Type"].value_counts()

In [ ]:
df["property_type"].value_counts()

In [ ]:
# Every listing has a unique property_id, so a plain df.duplicated() always returns 0.
# A stricter check ignores the ID and the URL.
print("Plain duplicates:", df.duplicated().sum())
print("Duplicates ignoring ID and URL:", df.drop(columns=["property_id", "page_url"]).duplicated().sum())

## 4. Keep only "For Sale" listings

Rent is a monthly amount and sale price is a total amount. Mixing them in one `price` column would confuse the model, and I want to predict sale prices, so I keep only "For Sale".

In [ ]:
before = len(df)
df = df[df["purpose"] == "For Sale"].copy()
summary["Rental listings removed"] = before - len(df)
print("Rows removed:", before - len(df), "| Rows now:", len(df))

## 5. Convert all areas to Marla

Areas come in two units. **1 Kanal = 20 Marla.** `Area Size` is the number in the unit given by `Area Type`, so a Kanal row must be multiplied by 20.

In [ ]:
df["area_marla"] = df["Area Size"].where(df["Area Type"] == "Marla", df["Area Size"] * 20)

In [ ]:
# Verification: Kanal rows must be exactly 20 times Area Size, Marla rows must be unchanged
kanal = df["Area Type"] == "Kanal"
print("Kanal rows correct:", (df.loc[kanal, "area_marla"] == df.loc[kanal, "Area Size"] * 20).all())
print("Marla rows correct:", (df.loc[~kanal, "area_marla"] == df.loc[~kanal, "Area Size"]).all())

In [ ]:
df["area_marla"].describe()

## 6. Convert `date_added` to a real date

The column is text. I parse it as month-day-year (for example `07-17-2019`) and add year and month columns.

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m-%d-%Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["date_added"].dtype

## 7. Hidden missing values: zero bedrooms and baths

`isnull()` shows no missing values in `bedrooms` and `baths`, but many rows contain 0. A house or flat with 0 bathrooms is not realistic, so 0 most likely means "not entered". I will treat these zeros as missing values later in this notebook.

In [ ]:
print("Bedrooms = 0:", (df["bedrooms"] == 0).sum())
print("Baths = 0:", (df["baths"] == 0).sum())

In [ ]:
df[df["bedrooms"] == 0]["property_type"].value_counts()

In [ ]:
df[df["baths"] == 0]["property_type"].value_counts()

## 8. Drop columns that do not help predict price

- `area`, `Area Type`, `Area Size`, `Area Category`: all replaced by `area_marla` (checked below that Area Category is only a size bucket)
- `agency`, `agent`, `page_url`: describe the seller or the listing, not the house (and 26% of agent and agency values are missing)
- `property_id`: an ID, no predictive value
- `purpose`: now the same value ("For Sale") in every row

In [ ]:
df["Area Category"].value_counts().head(10)

In [ ]:
cols_to_drop = ["area", "Area Type", "Area Size", "Area Category",
                "agency", "agent", "page_url", "purpose", "property_id"]
df = df.drop(columns=cols_to_drop)   # drop returns a new DataFrame, so it must be assigned back
df.columns.tolist()

## 9. Remove duplicate listings

The same house can be posted several times under different IDs. Duplicates inflate the data and can leak the same house into both the training and test sets, which makes accuracy look better than it is.

I compare every column except the dates, because a reposted house gets a new date.

In [ ]:
dup_cols = [c for c in df.columns if c not in ["date_added", "year_added", "month_added"]]
print("Duplicates found:", df.duplicated(subset=dup_cols).sum())

# Look at a sample before deciding
dups = df[df.duplicated(subset=dup_cols, keep=False)].sort_values(dup_cols)
dups[["property_type", "location", "price", "area_marla", "bedrooms", "baths", "latitude", "longitude"]].head(20)

**My decision:** (write here after looking at the sample above. Do the pairs look like the same listing posted twice, or different units in one project?)

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=dup_cols).copy()
summary["Duplicate listings removed"] = before - len(df)
print("Removed:", before - len(df), "| Rows now:", len(df))

## 10. Area outliers

`area_marla` has a minimum of 0 (impossible) and a maximum of 12,000 marla (600 Kanal), while the median is only about 6 marla. A few extreme values pull the mean far above the median.

**Rule:** keep houses between 1 and 100 marla (up to 5 Kanal), which covers normal residential properties.

In [ ]:
print("Zero area:", (df["area_marla"] == 0).sum())
print(df["area_marla"].quantile([0.01, 0.99, 0.999]))

In [ ]:
df.sort_values("area_marla", ascending=False)[["property_type", "city", "price", "area_marla"]].head(10)

In [ ]:
mask = df["area_marla"].between(1, 100)
print("Rows removed:", (~mask).sum())
before = len(df)
df = df[mask].copy()
summary["Area outliers removed"] = before - len(df)
df["area_marla"].describe()

## 11. Fill the zero bedrooms and baths

Deleting every row with a zero would throw away a large part of the data. Instead I turn the zeros into NaN, then fill them with the **median of the same property type**. The median is not affected by extreme values, and grouping by property type is fair because a flat and a farm house have different typical values. A flag column records which values were filled in.

In [ ]:
for col in ["bedrooms", "baths"]:
    df[col + "_was_missing"] = (df[col] == 0)
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df.groupby("property_type")[col].transform("median"))
    df[col] = df[col].fillna(df[col].median())      # safety net for a group with no valid value
    df[col] = df[col].round().astype(int)

df[["bedrooms", "baths"]].describe()

In [ ]:
df[["bedrooms", "baths"]].isnull().sum()

## 12. Price outliers

Prices that are impossibly low or high would pull the model in the wrong direction. First I look at the distribution and the extremes.

In [ ]:
df["price"].describe()

In [ ]:
df.sort_values("price")[["property_type", "city", "area_marla", "price"]].head(10)

In [ ]:
df.sort_values("price", ascending=False)[["property_type", "city", "area_marla", "price"]].head(10)

**Rule:** keep prices between the 1st and 99th percentile. (After running, check that the low value is believable for a house in Pakistan and the high value is believable for a large property.)

In [ ]:
low, high = df["price"].quantile([0.01, 0.99])
print(f"Low: {low:,.0f}  High: {high:,.0f}")
print("Rows removed:", (~df["price"].between(low, high)).sum())

before = len(df)
df = df[df["price"].between(low, high)].copy()
summary["Price outliers removed"] = before - len(df)

## 13. Final report and save

Final check of the cleaned data: how many rows were removed at each step, whether any values are missing, and what the table looks like.

In [ ]:
df = df.reset_index(drop=True)

report = pd.Series(summary, name="rows removed").to_frame()
report.loc["TOTAL removed"] = rows_before - len(df)
display(report)

print("Rows before:", rows_before, "| Rows after:", len(df))
print(f"Kept: {len(df) / rows_before:.1%} of the original data")
print("Total missing values:", df.isnull().sum().sum())

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.to_csv("../data/cleaned_house_prices.csv", index=False)
print("Saved:", df.shape)

## What I learned in this notebook

1. **Rent and sale prices cannot share one column.** They mean different things, so I kept only "For Sale".
2. **A number needs its unit.** `Area Size` is only meaningful together with `Area Type`. I converted everything to Marla (1 Kanal = 20 Marla) and verified the result.
3. **`df.drop()` returns a new DataFrame.** It must be assigned back: `df = df.drop(...)`.
4. **Missing values can hide as zeros.** `isnull()` reported none in bedrooms, but 0 was used for "not entered".
5. **A unique ID hides duplicates.** `df.duplicated()` returned 0 until I ignored `property_id`.
6. **After filtering, row numbers stay the same.** A `KeyError` from `loc` can mean the row was filtered out, not that the code is wrong.
7. **Run order matters.** I finish with Restart kernel and Run All so the notebook can be reproduced from top to bottom.